In [ ]:
import pandas as pd

df = pd.read_csv("/content/sunandini25187_iiitd.ac.in1695.tsv", sep="\t")
print(df.head())
print(df.columns)

   BindingDB Reactant_set_id  \
0                     370990   
1                     370989   
2                     370991   
3                     370993   
4                     370992   

                                       Ligand SMILES  \
0  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cc...   
1  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cc...   
2  COc1cnc(cn1)C(=O)Nc1ccc(F)c(c1)[C@]1(C)CS(=O)(...   
3  C[C@]1(CS(=O)(=O)C2(CCCC2)C(N)=N1)c1cc(NC(=O)c...   
4  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cn...   

                                        Ligand InChI  \
0  InChI=1S/C20H20FN5O3S/c1-19(2)18(23)26-20(3,11...   
1  InChI=1S/C19H20ClFN4O3S/c1-18(2)17(22)25-19(3,...   
2  InChI=1S/C19H22FN5O4S/c1-18(2)17(21)25-19(3,10...   
3  InChI=1S/C22H22FN5O3S/c1-21(13-32(30,31)22(20(...   
4  InChI=1S/C19H20F3N5O3S/c1-18(2)17(23)27-19(3,9...   

              Ligand InChI Key  BindingDB MonomerID  \
0  SQPMMXMVMZGIEG-FQEVSTJZSA-N               209917   
1  LMHDJRUKZXENER-IBGZP

/tmp/ipykernel_11924/1760773096.py:3: DtypeWarning: Columns (8,9,10,11,35,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/sunandini25187_iiitd.ac.in1695.tsv", sep="\t")


In [ ]:
# Keep only required columns
df = df[['Ligand SMILES', 'Ki (nM)', 'Target Name', 'BindingDB MonomerID']]

print(df.head())
print(df.shape)

                                       Ligand SMILES Ki (nM)  \
0  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cc...     NaN   
1  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cc...     NaN   
2  COc1cnc(cn1)C(=O)Nc1ccc(F)c(c1)[C@]1(C)CS(=O)(...     NaN   
3  C[C@]1(CS(=O)(=O)C2(CCCC2)C(N)=N1)c1cc(NC(=O)c...     NaN   
4  CC1(C)C(N)=N[C@@](C)(CS1(=O)=O)c1cc(NC(=O)c2cn...     NaN   

        Target Name  BindingDB MonomerID  
0  Beta-secretase 1               209917  
1  Beta-secretase 1             50012660  
2  Beta-secretase 1               209918  
3  Beta-secretase 1               209920  
4  Beta-secretase 1               209919  
(14546, 4)


In [ ]:
df = df.dropna(subset=['Ligand SMILES', 'Ki (nM)'])

In [ ]:
df = df[pd.to_numeric(df['Ki (nM)'], errors='coerce').notnull()]
df['Ki (nM)'] = df['Ki (nM)'].astype(float)

In [ ]:
df = df.drop_duplicates(subset='Ligand SMILES')

In [ ]:
import numpy as np

R = 1.987e-3
T = 298

# Convert nM to M first
df['Ki_M'] = df['Ki (nM)'] * 1e-9

# Calculate Delta G
df['Delta_G'] = R * T * np.log(df['Ki_M'])

In [ ]:
df_final = df[['Ligand SMILES', 'Ki (nM)', 'Delta_G', 'BindingDB MonomerID']]

print(df_final.head())
print(df_final.shape)

                                        Ligand SMILES  Ki (nM)    Delta_G  \
11  CC(C)CNC(=O)[C@@H](NC[C@H](Cc1ccccc1)NC(=O)c1c...    0.017 -14.683427   
16  CCC[C@H](O)[C@H](NC[C@H](Cc1ccccc1)NC(=O)c1cc2...    0.036 -14.239151   
26  CC(C)CNC(=O)[C@@H](NC(=O)[C@H](C)C[C@H](O)[C@H...    0.120 -13.526248   
29  CC(C)C[C@H](NC(=O)[C@H](CS(C)(=O)=O)NC(=O)OCc1...    0.120 -13.526248   
30  C[C@]1(C=CSC(=N1)N)c2cc(ccc2F)NC(=O)c3ccc(cn3)C#N    0.130 -13.478852   

    BindingDB MonomerID  
11             50398475  
16             50398472  
26                16254  
29             50231938  
30               210070  
(2303, 4)


In [ ]:
df_final.to_csv("BACE1_stage1_processed.csv", index=False)

Stage 2


In [ ]:
from sklearn.model_selection import train_test_split
splits = []

random_seeds = [42, 52, 62]

for seed in random_seeds:

    train_df, test_df = train_test_split(
        df_final,
        test_size=400,
        random_state=seed,
        shuffle=True
    )

    splits.append((train_df, test_df))

In [ ]:
for i, (train_df, test_df) in enumerate(splits):
    print(f"Split {i+1}")
    print("Train:", train_df.shape)
    print("Test :", test_df.shape)
    print()

Split 1
Train: (1903, 4)
Test : (400, 4)

Split 2
Train: (1903, 4)
Test : (400, 4)

Split 3
Train: (1903, 4)
Test : (400, 4)



In [ ]:
for i, (train_df, test_df) in enumerate(splits):

    train_df.to_csv(f"train_split_{i+1}.csv", index=False)
    test_df.to_csv(f"test_split_{i+1}.csv", index=False)

In [ ]:
from google.colab import files

for i in range(1,4):
    files.download(f"train_split_{i}.csv")
    files.download(f"test_split_{i}.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Stage 3 A (Random forect)


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 47.9 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 455, in run
    installed = install_given_reqs(
                ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/__init__.py", line 70, in install_given_reqs
    requirement.install(
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/req_install.py", line 851, in install
    install_wheel(
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/install/wheel.py", line 726, 

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

ImportError: libboost_log_setup-1e24bea2.so.1.85.0: cannot open shared object file: No such file or directory

In [ ]:
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(Descriptors._descList)
    descriptors = [descriptor[1](mol) for descriptor in Descriptors._descList]
    return pd.Series(descriptors, index=[descriptor[0] for descriptor in Descriptors._descList])

train_df = splits[0][0]
test_df = splits[0][1]

X_train_desc = train_df['Ligand SMILES'].apply(calculate_descriptors)
X_test_desc = test_df['Ligand SMILES'].apply(calculate_descriptors)

In [ ]:
descriptor_names = [descriptor[0] for descriptor in Descriptors._descList]

In [ ]:
print(X_train_desc.shape)

In [ ]:
zero_frac = (X_train_desc == 0).mean()

keep_cols = zero_frac[zero_frac < 0.90].index

X_train_desc = X_train_desc[keep_cols]
X_test_desc = X_test_desc[keep_cols]

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

X_train_desc = pd.DataFrame(
    imputer.fit_transform(X_train_desc),
    columns=X_train_desc.columns
)

X_test_desc = pd.DataFrame(
    imputer.transform(X_test_desc),
    columns=X_test_desc.columns
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_desc = scaler.fit_transform(X_train_desc)
X_test_desc = scaler.transform(X_test_desc)

In [ ]:
y_train = train_df['Delta_G'].values
y_test = test_df['Delta_G'].values

In [ ]:
print(X_train_desc.shape)
print(X_test_desc.shape)

In [ ]:
X_train_desc.shape = (1903,151)
X_test_desc.shape = (400,151)

In [ ]:
y_train.shape = (1903,)
y_test.shape = (400,)

In [ ]:
print(np.isnan(X_train_desc).sum())
print(np.isnan(X_test_desc).sum())

STAGE 3B — Train Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_desc, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test_desc)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)

pearson_corr = pearsonr(y_test, y_pred_rf)[0]
spearman_corr = spearmanr(y_test, y_pred_rf)[0]

print("RF Results")
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)
print("Pearson:", pearson_corr)
print("Spearman:", spearman_corr)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.6)

plt.plot(
    [min(y_test), max(y_test)],
    [min(y_test), max(y_test)],
    'r--'
)

plt.xlabel("Experimental ΔG")
plt.ylabel("Predicted ΔG")
plt.title("Random Forest Prediction Performance")
plt.show()

for all 3 replicates

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

results = []

for i, (train_df, test_df) in enumerate(splits):

    print(f"\nRunning Split {i+1}...")

    # Descriptor generation
    X_train_desc = train_df['Ligand SMILES'].apply(calculate_descriptors)
    X_test_desc = test_df['Ligand SMILES'].apply(calculate_descriptors)

    # The following lines are removed as X_train_desc and X_test_desc are already DataFrames
    # X_train_desc = pd.DataFrame(X_train_desc.tolist(), columns=descriptor_names)
    # X_test_desc = pd.DataFrame(X_test_desc.tolist(), columns=descriptor_names)

    # Remove sparse features
    zero_frac = (X_train_desc == 0).mean()
    keep_cols = zero_frac[zero_frac < 0.90].index

    X_train_desc = X_train_desc[keep_cols]
    X_test_desc = X_test_desc[keep_cols]

    # Remove correlated features
    corr_matrix = X_train_desc.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > 0.95)
    ]

    X_train_desc = X_train_desc.drop(columns=to_drop)
    X_test_desc = X_test_desc.drop(columns=to_drop)

    # Imputation
    imputer = SimpleImputer(strategy='median')

    X_train_desc = pd.DataFrame(
        imputer.fit_transform(X_train_desc),
        columns=X_train_desc.columns
    )

    X_test_desc = pd.DataFrame(
        imputer.transform(X_test_desc),
        columns=X_test_desc.columns
    )

    # Scaling
    scaler = StandardScaler()

    X_train_desc = scaler.fit_transform(X_train_desc)
    X_test_desc = scaler.transform(X_test_desc)

    # Targets
    y_train = train_df['Delta_G'].values
    y_test = test_df['Delta_G'].values

    # RF model
    rf_model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train_desc, y_train)

    y_pred = rf_model.predict(X_test_desc)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    pearson_corr = pearsonr(y_test, y_pred)[0]
    spearman_corr = spearmanr(y_test, y_pred)[0]

    results.append([rmse, mae, r2, pearson_corr, spearman_corr])

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=['RMSE','MAE','R2','Pearson','Spearman']
)

print(results_df)

In [ ]:
print("\nAverage Performance Across 3 Splits:\n")

for col in results_df.columns:

    mean = results_df[col].mean()
    std = results_df[col].std()

    print(f"{col}: {mean:.3f} ± {std:.3f}")

LR

In [ ]:
import deepchem as dc
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

featurizer = dc.feat.RDKitDescriptors()

lr_results = []

for i, (train_df, test_df) in enumerate(splits):

    print(f"\n===== LINEAR REGRESSION SPLIT {i+1} =====")

    # Descriptor generation
    X_train = featurizer.featurize(train_df['Ligand SMILES'])
    X_test = featurizer.featurize(test_df['Ligand SMILES'])

    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)

    # Remove sparse features
    zero_frac = (X_train == 0).mean()
    keep_cols = zero_frac[zero_frac < 0.90].index

    X_train = X_train[keep_cols]
    X_test = X_test[keep_cols]

    # Remove correlated features
    corr_matrix = X_train.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > 0.95)
    ]

    X_train = X_train.drop(columns=to_drop)
    X_test = X_test.drop(columns=to_drop)

    # Median imputation
    imputer = SimpleImputer(strategy='median')

    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    # Standardization
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Targets
    y_train = train_df['Delta_G'].values
    y_test = test_df['Delta_G'].values

    # Hyperparameter tuning
    param_grid = {
        'alpha': [0.01, 0.1, 1, 10, 100]
    }

    grid = GridSearchCV(
        Ridge(),
        param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    print("Best Alpha:", grid.best_params_)

    # Predict
    y_pred = best_model.predict(X_test)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    pearson_corr = pearsonr(y_test, y_pred)[0]
    spearman_corr = spearmanr(y_test, y_pred)[0]

    lr_results.append([rmse, mae, r2, pearson_corr, spearman_corr])

In [ ]:
lr_results_df = pd.DataFrame(
    lr_results,
    columns=['RMSE','MAE','R2','Pearson','Spearman']
)

print(lr_results_df)

In [ ]:
print("\nAverage Performance Across 3 Splits:\n")

for col in lr_results_df.columns:

    mean = lr_results_df[col].mean()
    std = lr_results_df[col].std()

    print(f"{col}: {mean:.3f} ± {std:.3f}")

XGBoost

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

xgb_results = []

for i, (train_df, test_df) in enumerate(splits):

    print(f"\n===== XGBOOST SPLIT {i+1} =====")

    # Descriptor generation
    X_train = featurizer.featurize(train_df['Ligand SMILES'])
    X_test = featurizer.featurize(test_df['Ligand SMILES'])

    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)

    # Remove sparse features
    zero_frac = (X_train == 0).mean()
    keep_cols = zero_frac[zero_frac < 0.90].index

    X_train = X_train[keep_cols]
    X_test = X_test[keep_cols]

    # Remove correlated features
    corr_matrix = X_train.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > 0.95)
    ]

    X_train = X_train.drop(columns=to_drop)
    X_test = X_test.drop(columns=to_drop)

    # Median imputation
    imputer = SimpleImputer(strategy='median')

    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    # Standardization
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Targets
    y_train = train_df['Delta_G'].values
    y_test = test_df['Delta_G'].values

    # Hyperparameter tuning
    param_grid = {
        'n_estimators': [100, 300],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1]
    }

    grid = GridSearchCV(
        XGBRegressor(
            objective='reg:squarederror',
            random_state=42
        ),
        param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    print("Best Params:", grid.best_params_)

    # Predict
    y_pred = best_model.predict(X_test)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    pearson_corr = pearsonr(y_test, y_pred)[0]
    spearman_corr = spearmanr(y_test, y_pred)[0]

    xgb_results.append([rmse, mae, r2, pearson_corr, spearman_corr])

In [ ]:
xgb_results_df = pd.DataFrame(
    xgb_results,
    columns=['RMSE','MAE','R2','Pearson','Spearman']
)

print(xgb_results_df)

In [ ]:
print("\nAverage Performance Across 3 Splits:\n")

for col in xgb_results_df.columns:

    mean = xgb_results_df[col].mean()
    std = xgb_results_df[col].std()

    print(f"{col}: {mean:.3f} ± {std:.3f}")

In [ ]:
import pandas as pd

# Add a 'Model' column to each results DataFrame
results_df['Model'] = 'Random Forest'
lr_results_df['Model'] = 'Linear Regression'
xgb_results_df['Model'] = 'XGBoost'

# Concatenate all results into a single DataFrame
all_results_df = pd.concat([lr_results_df, results_df, xgb_results_df], ignore_index=True)

# Reorder columns to have 'Model' first for better readability
all_results_df = all_results_df[['Model', 'RMSE', 'MAE', 'R2', 'Pearson', 'Spearman']]

print("\n--- Model Performance Across 3 Splits ---")
print(all_results_df)

In [ ]:
import pandas as pd

def get_avg_performance(results_df, model_name):
    avg_data = {'Model': model_name}
    for col in results_df.columns:
        if col != 'Model': # Exclude the 'Model' column if already present
            mean = results_df[col].mean()
            std = results_df[col].std()
            avg_data[col] = f"{mean:.3f} ± {std:.3f}"
    return pd.DataFrame([avg_data])


# Get average performance for each model
lr_avg = get_avg_performance(lr_results_df, 'Linear Regression')
rf_avg = get_avg_performance(results_df, 'Random Forest')
xgb_avg = get_avg_performance(xgb_results_df, 'XGBoost')

# Concatenate into a single DataFrame
overall_avg_performance = pd.concat([lr_avg, rf_avg, xgb_avg], ignore_index=True)

print("\n--- Average Model Performance Across 3 Splits ---")
display(overall_avg_performance)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
import joblib

joblib.dump(best_model, "best_xgb.pkl")
joblib.dump(imputer, "imputer.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(keep_cols, "keep_cols.pkl")

print("All files saved successfully!")

In [ ]:
import os
print(os.listdir())

In [ ]:
val2 = pd.read_excel("/content/Validation_set_2.csv.xlsx")

In [ ]:
!pip install deepchem
import deepchem as dc

featurizer = dc.feat.RDKitDescriptors()
X_val2 = featurizer.featurize(val2['SMILES'])
X_val2 = pd.DataFrame(X_val2)

In [ ]:
X_val2 = X_val2[keep_cols]

In [ ]:
X_val2 = X_val2.drop(columns=to_drop)

In [ ]:
X_val2 = imputer.transform(X_val2)
X_val2 = scaler.transform(X_val2)

In [ ]:
X_val2 = featurizer.featurize(val2['SMILES'])
X_val2 = pd.DataFrame(X_val2)

X_val2 = X_val2[keep_cols]

X_val2 = X_val2.drop(columns=to_drop)

X_val2 = imputer.transform(X_val2)

X_val2 = scaler.transform(X_val2)

In [ ]:
print(len(to_drop))

In [ ]:
X_val2 = featurizer.featurize(val2['SMILES'])
X_val2 = pd.DataFrame(X_val2)

X_val2 = X_val2[keep_cols]

X_val2 = X_val2.drop(columns=to_drop)

X_val2 = imputer.transform(X_val2)

X_val2 = scaler.transform(X_val2)

In [ ]:
type(X_val2)

In [ ]:
val2['Predicted_Delta_G'] = best_model.predict(X_val2)

In [ ]:
print(best_model.predict(X_val2))

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import joblib
import deepchem as dc
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#########################################
# LOAD SAVED OBJECTS (only best_model)
#########################################
best_model = joblib.load("best_xgb.pkl")

#########################################
# Re-define necessary objects
#########################################
# DeepChem featurizer was used for XGBoost training
featurizer = dc.feat.RDKitDescriptors()

#########################################
# Prepare X_test and y_test for the LAST split (Split 3)
# as best_model was trained on it.
#########################################
train_df, test_df = splits[2] # Use the last split (index 2 for split 3)
y_test = test_df['Delta_G'].values

# --- REGENERATE FEATURE SELECTION LOGIC FROM TRAINING ---
# This mirrors how X_train was prepared for best_model.fit in EfG-N7BqjW2i

# 1. Featurize training data
X_train_xgb_raw = featurizer.featurize(train_df['Ligand SMILES'])
X_train_xgb_df = pd.DataFrame(X_train_xgb_raw) # DataFrame with integer columns

# 2. Remove sparse features (find keep_cols_xgb_int)
zero_frac_train = (X_train_xgb_df == 0).mean()
keep_cols_xgb_int = zero_frac_train[zero_frac_train < 0.90].index # These are integer indices

X_train_xgb_df_filtered_sparse = X_train_xgb_df[keep_cols_xgb_int]

# 3. Remove correlated features (find to_drop_xgb_int)
corr_matrix_train = X_train_xgb_df_filtered_sparse.corr().abs()
upper_train = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))
to_drop_xgb_int = [column for column in upper_train.columns if any(upper_train[column] > 0.95)] # These are integer indices

# Apply all preprocessing to X_train_xgb_df to get the data shape it was fitted on
X_train_xgb_df_processed = X_train_xgb_df_filtered_sparse.drop(columns=to_drop_xgb_int)

# Instantiate and fit NEW imputer and scaler for the XGBoost pipeline
imputer = SimpleImputer(strategy='median')
scholer = StandardScaler() # Typo 'scholer' from previous iteration, corrected to 'scaler'

# Fit imputer and scaler on the processed training data for XGBoost
X_train_imputed = imputer.fit_transform(X_train_xgb_df_processed)
scholer.fit(X_train_imputed)

# --- APPLY THE SAME LOGIC TO X_TEST ---

# Generate descriptors for test set
X_test_raw = featurizer.featurize(test_df['Ligand SMILES'])
X_test_df = pd.DataFrame(X_test_raw) # DataFrame with integer columns

# Apply sparse feature removal
X_test_df_filtered_sparse = X_test_df[keep_cols_xgb_int]

# Apply correlated feature removal
X_test_df_processed = X_test_df_filtered_sparse.drop(columns=to_drop_xgb_int)

# Convert to NumPy array for imputer and scaler
X_test = X_test_df_processed.values

# Impute and Scale X_test using the newly fitted imputer and scaler
X_test = imputer.transform(X_test)
X_test = scholer.transform(X_test)

#########################################
# Predict on Test Set
#########################################
y_pred = best_model.predict(X_test)

#########################################
# Calculate Pearson Correlation
#########################################
pearson_corr, p_value = pearsonr(y_test, y_pred)

print(f"Pearson Correlation: {pearson_corr:.4f}")
print(f"P-value: {p_value:.4e}")

#########################################
# Scatter Plot
#########################################
plt.figure(figsize=(7,7))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.7,
    edgecolor='black'
)

#########################################
# Identity Line
#########################################
min_val = min(min(y_test), min(y_pred))
max_val = max(max(y_test), max(y_pred))

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    'r--',
    linewidth=2
)

#########################################
# Labels
#########################################
plt.xlabel("Experimental ΔG", fontsize=12)
plt.ylabel("Predicted ΔG", fontsize=12)
plt.title(f"XGBoost Test Set\nPearson r = {pearson_corr:.3f}", fontsize=14)

#########################################
# Grid
#########################################
plt.grid(True)

plt.show()

In [ ]:
!pip install rdkit

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
import joblib

#########################################
# LOAD SAVED OBJECTS
#########################################
best_model = joblib.load("best_xgb.pkl")
imputer = joblib.load("imputer.pkl")
scaler = joblib.load("scaler.pkl")
keep_cols = joblib.load("keep_cols.pkl")

#########################################
# LOAD YOUR DATA
#########################################
my_data = pd.read_excel("DATA_1.xlsx")

#########################################
# FUNCTION TO CALCULATE RDKit DESCRIPTORS
#########################################
descriptor_names = [desc[0] for desc in Descriptors._descList]

def calc_rdkit_descriptors(smiles):
    # Ensure the input is a string before passing to Chem.MolFromSmiles
    if not isinstance(smiles, str):
        return [np.nan]*len(descriptor_names)

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [np.nan]*len(descriptor_names)

    return [func(mol) for name, func in Descriptors._descList]

#########################################
# GENERATE DESCRIPTORS
#########################################
X_new = my_data['Smiles'].apply(calc_rdkit_descriptors)

X_new = pd.DataFrame(
    X_new.tolist(),
    columns=descriptor_names
)

#########################################
# KEEP SAME TRAINING FEATURES (Sparse Features)
#########################################
# keep_cols already contains the names of the non-sparse features
X_new = X_new[keep_cols]

#########################################
# REGENERATE 'to_drop' FOR CORRELATED FEATURES
#########################################
# Load the training data used for the best model to recreate 'to_drop'
# Assuming best_xgb.pkl was trained on the last split (split 3)
train_df_for_corr = pd.read_csv("train_split_3.csv")

X_train_desc_for_corr = train_df_for_corr['Ligand SMILES'].apply(calc_rdkit_descriptors)
X_train_desc_for_corr = pd.DataFrame(
    X_train_desc_for_corr.tolist(),
    columns=descriptor_names
)
X_train_desc_for_corr = X_train_desc_for_corr[keep_cols] # Apply initial sparse feature filter

# Calculate correlation matrix and identify features to drop
corr_matrix = X_train_desc_for_corr.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

# Apply 'to_drop' to X_new
X_new = X_new.drop(columns=to_drop)

#########################################
# IMPUTE
#########################################
X_new = imputer.transform(X_new)

#########################################
# SCALE
#########################################
X_new = scaler.transform(X_new)

#########################################
# PREDICT
#########################################
my_data['Predicted_Delta_G'] = best_model.predict(X_new)

#########################################
# SAVE
#########################################
my_data.to_excel("Predicted_DeltaG.xlsx", index=False)

print(my_data[['Smiles','Predicted_Delta_G']].head())

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import joblib
import deepchem as dc
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#########################################
# LOAD SAVED OBJECTS (only best_model)
#########################################
best_model = joblib.load("best_xgb.pkl")

#########################################
# Re-define necessary objects
#########################################
# DeepChem featurizer was used for XGBoost training
featurizer = dc.feat.RDKitDescriptors()

#########################################
# Prepare X_test and y_test for the LAST split (Split 3)
# as best_model was trained on it.
#########################################
train_df, test_df = splits[2] # Use the last split (index 2 for split 3)
y_test = test_df['Delta_G'].values

# --- REGENERATE FEATURE SELECTION LOGIC FROM TRAINING ---
# This mirrors how X_train was prepared for best_model.fit in EfG-N7BqjW2i

# 1. Featurize training data
X_train_xgb_raw = featurizer.featurize(train_df['Ligand SMILES'])
X_train_xgb_df = pd.DataFrame(X_train_xgb_raw) # DataFrame with integer columns

# 2. Remove sparse features (find keep_cols_xgb_int)
zero_frac_train = (X_train_xgb_df == 0).mean()
keep_cols_xgb_int = zero_frac_train[zero_frac_train < 0.90].index # These are integer indices

X_train_xgb_df_filtered_sparse = X_train_xgb_df[keep_cols_xgb_int]

# 3. Remove correlated features (find to_drop_xgb_int)
corr_matrix_train = X_train_xgb_df_filtered_sparse.corr().abs()
upper_train = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))
to_drop_xgb_int = [column for column in upper_train.columns if any(upper_train[column] > 0.95)] # These are integer indices

# Apply all preprocessing to X_train_xgb_df to get the data shape it was fitted on
X_train_xgb_df_processed = X_train_xgb_df_filtered_sparse.drop(columns=to_drop_xgb_int)

# Instantiate and fit NEW imputer and scaler for the XGBoost pipeline
imputer = SimpleImputer(strategy='median')
scholer = StandardScaler()

# Fit imputer and scaler on the processed training data for XGBoost
X_train_imputed = imputer.fit_transform(X_train_xgb_df_processed)
scholer.fit(X_train_imputed)

# --- APPLY THE SAME LOGIC TO X_TEST ---

# Generate descriptors for test set
X_test_raw = featurizer.featurize(test_df['Ligand SMILES'])
X_test_df = pd.DataFrame(X_test_raw) # DataFrame with integer columns

# Apply sparse feature removal
X_test_df_filtered_sparse = X_test_df[keep_cols_xgb_int]

# Apply correlated feature removal
X_test_df_processed = X_test_df_filtered_sparse.drop(columns=to_drop_xgb_int)

# Convert to NumPy array for imputer and scaler
X_test = X_test_df_processed.values

# Impute and Scale X_test using the newly fitted imputer and scaler
X_test = imputer.transform(X_test)
X_test = scholer.transform(X_test)

#########################################
# Predict on Test Set
#########################################
y_pred = best_model.predict(X_test)

#########################################
# Calculate Pearson Correlation
#########################################
pearson_corr, p_value = pearsonr(y_test, y_pred)

print(f"Pearson Correlation: {pearson_corr:.4f}")
print(f"P-value: {p_value:.4e}")

#########################################
# Scatter Plot
#########################################
plt.figure(figsize=(7,7))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.7,
    edgecolor='black'
)

#########################################
# Identity Line
#########################################
min_val = min(min(y_test), min(y_pred))
max_val = max(max(y_test), max(y_pred))

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    'r--',
    linewidth=2
)

#########################################
# Labels
#########################################
plt.xlabel("Experimental ΔG", fontsize=12)
plt.ylabel("Predicted ΔG", fontsize=12)
plt.title(f"XGBoost Test Set\nPearson r = {pearson_corr:.3f}", fontsize=14)

#########################################
# Grid
#########################################
plt.grid(True)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import deepchem as dc
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#########################################
# LOAD MODEL
#########################################
best_model = joblib.load("best_xgb.pkl")

#########################################
# LOAD YOUR FILE
#########################################
my_data = pd.read_excel("Processed_DATA_1.xlsx")

#########################################
# FEATURIZER
#########################################
featurizer = dc.feat.RDKitDescriptors()

#########################################
# TRAINING PREPROCESSING REBUILD
#########################################
train_df, test_df = splits[2]

X_train_raw = featurizer.featurize(train_df['Ligand SMILES'])
X_train_df = pd.DataFrame(X_train_raw)

zero_frac = (X_train_df == 0).mean()
keep_cols = zero_frac[zero_frac < 0.90].index

X_train_df = X_train_df[keep_cols]

corr_matrix = X_train_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

X_train_df = X_train_df.drop(columns=to_drop)

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train_df)
scaler.fit(X_train_imputed)

#########################################
# PROCESS YOUR SMILES
#########################################
X_new_raw = featurizer.featurize(my_data['Smiles'])
X_new_df = pd.DataFrame(X_new_raw)

X_new_df = X_new_df[keep_cols]
X_new_df = X_new_df.drop(columns=to_drop)

X_new = imputer.transform(X_new_df)
X_new = scaler.transform(X_new)

#########################################
# PREDICT DELTA G
#########################################
my_data['Predicted_Delta_G'] = best_model.predict(X_new)

#########################################
# CONVERT DELTA G TO Ki
#########################################
R = 1.987e-3
T = 298

my_data['Predicted_Ki_Molar'] = np.exp(
    my_data['Predicted_Delta_G']/(R*T)
)

#########################################
# Convert to nM
#########################################
my_data['Predicted_Ki_nM'] = (
    my_data['Predicted_Ki_Molar']*1e9
)

#########################################
# SAVE
#########################################
my_data.to_excel("Predicted_Ki_Output.xlsx", index=False)

print(my_data.head())

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_excel("DATA_1.xlsx")

print(df.head())
print(df.columns)

In [ ]:
import pandas as pd
import requests
import time

# Load your CHEMBL IDs from txt file
with open("/content/ChembleIDS.txt", "r") as f:
    chembl_ids = [line.strip() for line in f if line.strip()]

results = []

for chembl_id in chembl_ids:
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity.json?molecule_chembl_id={chembl_id}"

    try:
        response = requests.get(url)
        data = response.json()

        found = False
        for act in data.get("activities", []):
            if act.get("standard_type") == "Ki":
                results.append({
                    "CHEMBL_ID": chembl_id,
                    "Ki": act.get("standard_value"),
                    "Units": act.get("standard_units"),
                    "Target": act.get("target_chembl_id")
                })
                found = True

        if not found:
            results.append({
                "CHEMBL_ID": chembl_id,
                "Ki": None,
                "Units": None,
                "Target": None
            })

        time.sleep(0.2)

    except:
        results.append({
            "CHEMBL_ID": chembl_id,
            "Ki": "ERROR",
            "Units": None,
            "Target": None
        })

# Save results
df = pd.DataFrame(results)
df.to_csv("CHEMBL_Ki_values.csv", index=False)

print("Done! Saved as CHEMBL_Ki_values.csv")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

# Load predicted and actual files
pred_df = pd.read_excel("Predicted_Ki_Output.xlsx")
actual_df = pd.read_csv("CHEMBL_Ki_values.csv")

# Merge using CHEMBL ID
merged = pred_df.merge(actual_df,
                       left_on="Molecule ChEMBL ID",
                       right_on="CHEMBL_ID")

# Remove missing Ki
merged = merged.dropna(subset=["Ki"])

# Convert Ki nM → Molar
merged["Ki_Molar_Actual"] = merged["Ki"].astype(float) * 1e-9

# Calculate Experimental Delta G
R = 1.987
T = 298

merged["Actual_Delta_G"] = (R*T*np.log(merged["Ki_Molar_Actual"])) / 1000

# Pearson Correlation
pearson_corr, pval = pearsonr(
    merged["Predicted_Delta_G"],
    merged["Actual_Delta_G"]
)

# Spearman Correlation
spearman_corr, sp_p = spearmanr(
    merged["Predicted_Delta_G"],
    merged["Actual_Delta_G"]
)

print("Pearson Correlation:", pearson_corr)
print("Spearman Correlation:", spearman_corr)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    merged["Actual_Delta_G"],
    merged["Predicted_Delta_G"],
    alpha=0.7
)

plt.xlabel("Experimental ΔG")
plt.ylabel("Predicted ΔG")
plt.title(f"Predicted vs Experimental ΔG\nPearson r={pearson_corr:.3f}")

# Diagonal ideal line
lims = [
    min(merged["Actual_Delta_G"].min(), merged["Predicted_Delta_G"].min()),
    max(merged["Actual_Delta_G"].max(), merged["Predicted_Delta_G"].max())
]
plt.plot(lims, lims, 'r--')

plt.grid()
plt.show()

NameError: name 'plt' is not defined